In [1]:
import numpy as np
import pandas as pd
from itertools import combinations

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score

from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier

from imblearn.over_sampling import SMOTE

## Load Data

In [2]:
df = pd.read_csv('disease_dataset.csv')

le = LabelEncoder()
df['disease'] = le.fit_transform(df['disease'])

X = df.drop('disease', axis=1)
y = df['disease']

## Split + SMOTE

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

if y_train.value_counts().min() >= 2:
    X_train, y_train = SMOTE(random_state=42).fit_resample(X_train, y_train)

## Feature Engineering (FIXED)

In [4]:
def add_features(df, base_cols, top_cols=None):
    df = df.copy()
    df['symptom_count'] = df[base_cols].sum(axis=1)

    if top_cols is None:
        top_cols = df[base_cols].sum().sort_values(ascending=False).head(10).index.tolist()

    for c1, c2 in combinations(sorted(top_cols), 2):
        name = "_".join(sorted([c1, c2]))
        df[name] = df[c1] & df[c2]

    return df, top_cols

base_cols = X.columns

X_train, top_cols = add_features(X_train, base_cols)
X_test, _ = add_features(X_test, base_cols, top_cols)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

## Models

In [5]:
models = {
    "NB": BernoulliNB(alpha=0.1),
    "RF": RandomForestClassifier(n_estimators=200, random_state=42),
    "KNN": KNeighborsClassifier(metric='hamming'),
    "ENS": VotingClassifier(
        estimators=[
            ('nb', BernoulliNB(alpha=0.1)),
            ('rf', RandomForestClassifier(n_estimators=200)),
            ('knn', KNeighborsClassifier(metric='hamming'))
        ],
        voting='soft'
    )
}

## Train + Select Best Model

In [6]:
results = []
trained = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model

    proba = model.predict_proba(X_test)
    top3 = top_k_accuracy_score(y_test, proba, k=3)

    results.append((name, top3))

# Select best
best_model_name, best_acc = sorted(results, key=lambda x: x[1], reverse=True)[0]
best_model = trained[best_model_name]

print("Best Model Accuracy (Top-3):", round(best_acc, 2))

Best Model Accuracy (Top-3): 0.67


## Prediction Function

In [10]:
symptom_index = {col: i for i, col in enumerate(X.columns)}

def predict_disease(symptoms):
    # Create binary input vector
    data = np.zeros(len(symptom_index), dtype=int)

    unknown = []
    for s in symptoms:
        if s in symptom_index:
            data[symptom_index[s]] = 1
        else:
            unknown.append(s)

    if unknown:
        print("Ignored unknown symptoms:", unknown)

    # Convert to DataFrame
    df_input = pd.DataFrame([data], columns=X.columns)

    # Apply same feature engineering
    df_input, _ = add_features(df_input, X.columns, top_cols)

    # Align columns with training data
    df_input = df_input.reindex(columns=X_train.columns, fill_value=0)

    # Predict probabilities
    proba = best_model.predict_proba(df_input)[0]
    top3_idx = np.argsort(proba)[-3:][::-1]

    # Return exact required format
    return [(le.inverse_transform([i])[0], proba[i]) for i in top3_idx]

In [11]:
predict_disease(['fever','headache'])

[('Pneumonia', np.float64(0.956870058211108)),
 ('Impetigo', np.float64(0.020772878383920793)),
 ('Diabetes', np.float64(0.010341989318649237))]